# 🤖 Hiver AI Support Agent — @MicrosoftHelps

**Brand:** `microsofthelps` | **Dataset:** Kaggle Customer Support on Twitter

**Pipeline:** Intent Classification → Hybrid Retrieval → Escalation Gate → Reply Drafting → Safety Guardrails

---
## Before you start
1. Runtime → Change runtime type → **T4 GPU**
2. Click the 🔑 **Secrets** icon (left sidebar) → add `GROQ_API_KEY` and `HF_TOKEN`
3. In the next cell, set `HF_USERNAME` to your HuggingFace username
4. Run all cells top to bottom


---
## §0 · Setup

In [1]:
# ══════════════════════════════════════════════════════
# THE ONLY LINE YOU NEED TO EDIT
HF_USERNAME = 'SpongeBob387'
# ══════════════════════════════════════════════════════

In [2]:
# Install dependencies
!pip install -q \
    "transformers>=4.43.0" \
    "datasets>=2.19.0" \
    "peft>=0.11.0" \
    "bitsandbytes>=0.43.1" \
    "accelerate>=0.30.0" \
    "trl>=0.9.6" \
    "evaluate>=0.4.2" \
    "huggingface_hub>=0.23.4" \
    "sentence-transformers>=3.0.1" \
    "faiss-cpu>=1.8.0" \
    "rank-bm25>=0.2.2" \
    "groq>=0.9.0" \
    "scipy>=1.13.0" \
    "scikit-learn>=1.4.2" \
    "pyarrow>=15.0.0" \
    "emoji>=0.6.0"
print('✅ Dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 51.8 MB/s eta 0:00:00
✅ Dependencies installed


In [55]:
# Mount Drive + configure all paths + load secrets
import os, sys
from google.colab import drive, userdata

drive.mount('/content/drive')

# ── Paths (matches your Drive layout exactly) ──────────────
DRIVE_ROOT     = '/content/drive/MyDrive/Hiver/'
CODE_DIR       = DRIVE_ROOT + 'code/'          # pipeline/, evaluation/, data/ live here
DATA_DIR       = DRIVE_ROOT + 'data/'          # ms_threads.csv etc live here
INDEX_DIR      = DRIVE_ROOT + 'index/'         # built by §3 — auto-created
GOLDEN_SET_DIR = DRIVE_ROOT + 'golden_set/'    # built by §7 — auto-created
LOG_DIR        = DRIVE_ROOT + 'logs/'          # trace logs  — auto-created

# Make pipeline/evaluation/data importable
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# HuggingFace model ids (built from HF_USERNAME set above)
INTENT_MODEL_ID = f'{HF_USERNAME}/bertweet-mshelps-intent'
DRAFTER_BASE_ID = 'meta-llama/Llama-3.2-3B-Instruct'
DRAFTER_LORA_ID = f'{HF_USERNAME}/llama32-mshelps-drafter'

# Load secrets from Colab Secrets panel
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
os.environ['HF_TOKEN']     = userdata.get('HF_TOKEN')

# Seed
import random, numpy as np, torch
SEED, SUBSAMPLE_N, TOP_K = 42, 500, 5
random.seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.manual_seed(SEED)

print('✅ Drive mounted')
print(f'   CODE_DIR  : {CODE_DIR}')
print(f'   DATA_DIR  : {DATA_DIR}')
print(f'   GPU       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None — switch to T4"}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted
   CODE_DIR  : /content/drive/MyDrive/Hiver/code/
   DATA_DIR  : /content/drive/MyDrive/Hiver/data/
   GPU       : NVIDIA A100-SXM4-40GB


---
## §1 · Load Data & Demonstrate Thread Stitcher

In [4]:
import pandas as pd

threads_df    = pd.read_csv(DATA_DIR + 'ms_threads.csv', low_memory=False)
first_inbound = pd.read_csv(DATA_DIR + 'microsofthelps_first_inbound_k9.csv', low_memory=False)

print(f'threads_df    : {threads_df.shape}')
print(f'first_inbound : {first_inbound.shape}')
print(f'Columns (threads): {threads_df.columns.tolist()}')
threads_df.head(3)

threads_df    : (24514, 8)
first_inbound : (4458, 5)
Columns (threads): ['thread_id', 'turn_pos', 'tweet_id', 'author_id', 'inbound', 'text', 'parent', 'cluster']


,thread_id,turn_pos,tweet_id,author_id,inbound,text,parent,cluster
0,0,0,205,115751,True,"@MicrosoftHelps hi, having trouble making purc...",0,5
1,0,1,203,microsofthelps,False,"@115751 Hello, Josh! To get better assistance ...",205,5
2,0,2,206,microsofthelps,False,"@115751 Hello, Josh! Please help us improve ou...",205,5


In [5]:
# Demonstrate the thread stitcher on a real multi-part tweet
from pipeline.phase1_intake import stitch_multipart, prepare_thread_for_embedding

multipart = threads_df[threads_df['text'].str.contains(r'1/2|2/2', na=False, regex=True)]
if not multipart.empty:
    sample_id = multipart['thread_id'].iloc[0]
    turns = threads_df[threads_df['thread_id'] == sample_id].to_dict('records')
    print(f'Thread {sample_id} — {len(turns)} raw turns:')
    for t in turns:
        label = 'customer' if t.get('inbound', True) else 'brand'
        print(f'  [{label}] {str(t["text"])[:100]}')
    print()
    stitched = stitch_multipart(turns, author_key='inbound', text_key='text')
    print(f'After stitching — {len(stitched)} logical turns:')
    for t in stitched:
        label = 'customer' if t.get('inbound', True) else 'brand'
        print(f'  [{label}] {str(t["text"])[:140]}')
else:
    print('No multi-part threads found in this sample.')

Thread 10 — 6 raw turns:
  [customer] @118334 At long last app downloaded, clicked Install, now hung on "Just a moment...". My net cnxn fi
  [brand] @118333 2/2 Store app or via Microsoft Store online?
  [brand] @118333 1/2 Hi there, Bill! Were here to help. Just to clarify, where are you trying yo download an 
  [customer] @MicrosoftHelps Finally downloaded app but hangs on "Just a moment" then times out.
  [brand] @118333 Hello, Bill. Let us know on how we can improve our support here: https://t.co/ofGxB7VRWk. Th
  [brand] @118333 That's strange. It'd be best if you contact Skype team for them to have this checked: https:

After stitching — 6 logical turns:
  [customer] @118334 At long last app downloaded, clicked Install, now hung on "Just a moment...". My net cnxn fine. U have server problems?
  [brand] @118333 2/2 Store app or via Microsoft Store online?
  [brand] @118333 1/2 Hi there, Bill! Were here to help. Just to clarify, where are you trying yo download an app? Is it from th

---
## §2 · Intent Classifier (BERTweet-base)

In [56]:
import importlib
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from transformers import Trainer

import pipeline.phase2_intent
importlib.reload(pipeline.phase2_intent)

from pipeline.phase2_intent import (
    IntentClassifier,
    build_training_dataset,
    get_training_args,
    LABEL2ID,
)
from data.prepare_golden_set import CLUSTER_TO_CLASS
from evaluation.metrics import compute_classification_metrics, print_classification_report

print("Loading first_inbound data...")
first_inbound = pd.read_csv(DATA_DIR + 'microsofthelps_first_inbound_k9.csv', low_memory=False)
print(f"Columns: {first_inbound.columns.tolist()}")

# ── 1. Prepare intent labels from 'cluster_final' ──────────────────────────────
fi = first_inbound.copy()
fi['intent_class']    = fi['cluster_final'].map(CLUSTER_TO_CLASS).fillna('Windows General')
fi['intent_label_id'] = fi['intent_class'].map(LABEL2ID).fillna(0).astype(int)

# Use 'clean' if available, otherwise fallback to 'text'
text_col = 'clean' if 'clean' in fi.columns else 'text'

# Drop null or empty texts
fi = fi.dropna(subset=[text_col, 'intent_label_id'])
fi = fi[fi[text_col].astype(str).str.strip().str.len() > 0]

print("\nLabel distribution across the 8 classes:")
print(fi[['intent_class', 'intent_label_id']].value_counts().to_string())

# ── 2. Build train/val/test splits ───────────────────────────────────────────
dataset = build_training_dataset(
    fi,
    text_col=text_col,
    label_col='intent_label_id'
)
print(f"\nDataset splits:\n{dataset}")

# ── 3. Tokenize ──────────────────────────────────────────────────────────────
classifier = IntentClassifier.for_training(num_labels=8)

def tokenize(batch):
    return classifier.tokenizer(
        batch['text'], truncation=True, max_length=128, padding='max_length'
    )

dataset = dataset.map(tokenize, batched=True)

# ── 4. Trainer & Metrics (keyed as 'f1_macro' to match metric_for_best_model) ─
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = float(f1_score(labels, preds, average='macro', zero_division=0))
    return {'f1_macro': macro_f1}

trainer = Trainer(
    model=classifier.model,
    args=get_training_args(num_epochs=4, batch_size=32),
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    compute_metrics=compute_metrics,
)
trainer.train()

# ── 5. Evaluate on held-out test split ───────────────────────────────────────
print("\nEvaluating on held-out test set...")
test_preds_raw = trainer.predict(dataset['test'])
test_preds = np.argmax(test_preds_raw.predictions, axis=-1)
test_labels = test_preds_raw.label_ids

metrics = compute_classification_metrics(
    test_labels.tolist(),
    test_preds.tolist(),
    labels=list(range(8))
)
print_classification_report(metrics)

# ── 6. Push to HuggingFace Hub ───────────────────────────────────────────────
classifier.push_to_hub(INTENT_MODEL_ID)
print(f"\n✅ Fine-tuned model pushed to: https://huggingface.co/{INTENT_MODEL_ID}")

Loading first_inbound data...
Columns: ['thread_id', 'text', 'clean', 'cluster_k9', 'cluster_final']

Label distribution across the 8 classes:
intent_class                intent_label_id
Windows General             0                  2020
Windows Update & Install    1                  1006
Office & Productivity       2                   316
Account & Sign-In           3                   299
BSOD & Crash / Distress     5                   264
Surface Hardware            4                   201
Support Channel Navigation  7                   186
Xbox & Gaming               6                   141

Dataset splits:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 3324
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 444
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 665
    })
})
[IntentClassifier] Initializing base model for training: vinai/bertweet-base


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/3324 [00:00<?, ? examples/s]

Map:   0%|          | 0/444 [00:00<?, ? examples/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...3mcjc83/model.safetensors:   0%|          |  553kB /  540MB            

No files have been modified since last commit. Skipping to prevent empty commit.


[IntentClassifier] ✅ Model pushed to: https://huggingface.co/SpongeBob387/bertweet-mshelps-intent

✅ Fine-tuned model pushed to: https://huggingface.co/SpongeBob387/bertweet-mshelps-intent


In [8]:
import torch
from pipeline.phase2_intent import IntentClassifier, ID2LABEL

sample_tweet = "my xbox wont connect to xbox live since yesterday"
inputs = classifier.tokenizer(sample_tweet, return_tensors="pt", truncation=True, max_length=128)
inputs = {k: v.to(classifier.device) for k, v in inputs.items()}
with torch.no_grad():
    logits = classifier.model(**inputs).logits[0]
    probs  = torch.softmax(logits, dim=-1).cpu().numpy()

print("RAW PROBABILITY DISTRIBUTION (should spread across classes):")
for i, p in enumerate(probs):
    bar = "█" * int(p * 40)
    print(f"  {ID2LABEL[i]:<35} {p:.4f}  {bar}")

print()
if probs.max() > 0.85 and probs.argmax() == 0:
    print("⚠️  DIAGNOSIS: Model collapsed to majority class (Windows General).")
    print("   Root cause: training labels were likely all mapped to class 0.")
    print("   Fix: re-run §2-TRAIN with the corrected label mapping below.")
else:
    print("✅ Model looks healthy — probabilities spread across classes.")

# ── 2. Full smoke test — one example per intent class ───────────────────────
print("\n" + "=" * 65)
print("SMOKE TEST — one tweet per intent class")
print("=" * 65)

smoke_tests = [
    # (tweet,                                                     expected intent)
    ("my laptop keeps freezing and i dont know why",             "Windows General"),
    ("windows 10 update stuck at 35 percent for 3 hours",        "Windows Update & Install"),
    ("outlook wont open and excel keeps crashing on startup",     "Office & Productivity"),
    ("i cant log into my microsoft account forgot my password",   "Account & Sign-In"),
    ("my surface pro pen has completely stopped working",         "Surface Hardware"),
    ("got a blue screen of death bsod every hour please help",   "BSOD & Crash / Distress"),
    ("my xbox one wont connect to xbox live keeps disconnecting", "Xbox & Gaming"),
    ("how do i contact microsoft support what is the phone number", "Support Channel Navigation"),
    ("my xbox wont connect to xbox live since yesterday", "Xbox & Gaming"),
]

correct = 0
for tweet, expected in smoke_tests:
    pred = classifier.predict(tweet)
    match = "✅" if pred.label == expected else "❌"
    print(f"{match} Pred: {pred.label:<35} (conf={pred.confidence:.2f})")
    print(f"   Exp:  {expected}")
    print(f"   Tweet: {tweet[:70]}")
    if pred.label == expected:
        correct += 1

print(f"\nAccuracy on smoke test: {correct}/{len(smoke_tests)}")
if correct < 4:
    print("⚠️  Accuracy too low — classifier needs retraining. See fix below.")


RAW PROBABILITY DISTRIBUTION (should spread across classes):
  Windows General                     0.0357  █
  Windows Update & Install            0.0481  █
  Office & Productivity               0.0592  ██
  Account & Sign-In                   0.0826  ███
  Surface Hardware                    0.0638  ██
  BSOD & Crash / Distress             0.0682  ██
  Xbox & Gaming                       0.5972  ███████████████████████
  Support Channel Navigation          0.0451  █

✅ Model looks healthy — probabilities spread across classes.

SMOKE TEST — one tweet per intent class
✅ Pred: Windows General                     (conf=0.93)
   Exp:  Windows General
   Tweet: my laptop keeps freezing and i dont know why
✅ Pred: Windows Update & Install            (conf=0.92)
   Exp:  Windows Update & Install
   Tweet: windows 10 update stuck at 35 percent for 3 hours
✅ Pred: Office & Productivity               (conf=0.65)
   Exp:  Office & Productivity
   Tweet: outlook wont open and excel keeps crashing

---
## §3 · Build RAG Index (BM25 + FAISS)

In [9]:
# Skipped automatically if index already exists in Drive
import os
from data.build_rag_index import run_in_colab as build_index

index_exists = os.path.exists(INDEX_DIR + 'faiss_index.bin')

if index_exists:
    print(f'✅ Index already exists at {INDEX_DIR} — skipping build')
else:
    print('🔨 Building RAG index...')
    os.makedirs(INDEX_DIR, exist_ok=True)
    result = build_index(
        threads_csv=DATA_DIR + 'ms_threads.csv',
        first_inbound_csv=DATA_DIR + 'microsofthelps_first_inbound_k9.csv',
        output_dir=INDEX_DIR,
        max_turns=6,
        batch_size=64,
    )
    print(result['stats'])

✅ Index already exists at /content/drive/MyDrive/Hiver/index/ — skipping build


In [10]:
# Load retriever
from pipeline.phase3_retrieval import HybridRetriever

retriever = HybridRetriever.load(
    index_dir=INDEX_DIR,
    embedder_name='sentence-transformers/all-MiniLM-L6-v2',
)

# Smoke test
test_query = 'windows 10 update stuck at 30 percent for hours'
results = retriever.retrieve(test_query, intent_label='Windows Update & Install', top_k=3)
print(f'Query: {test_query}')
print(f'Retrieval confidence: {retriever.score_confidence(results):.3f}')
for r in results:
    print(f'  [RRF={r.rrf_score:.4f}] {r.thread.short_summary()[:100]}')

[HybridRetriever] Loading BM25 index from /content/drive/MyDrive/Hiver/index/bm25_index.pkl
[HybridRetriever] Loading FAISS index from /content/drive/MyDrive/Hiver/index/faiss_index.bin
[HybridRetriever] Loading metadata from /content/drive/MyDrive/Hiver/index/thread_metadata.json
[HybridRetriever] Loading embedder: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[HybridRetriever] ✅ Index loaded: 4458 threads
Query: windows 10 update stuck at 30 percent for hours
Retrieval confidence: 0.938
  [RRF=0.0308] customer: @MicrosoftHelps hello I’ve done it many times but still it get stuck during windows update
  [RRF=0.0303] customer: @MicrosoftHelps  every time I try updating Windows 10 I keep getting this. Last time it wa
  [RRF=0.0291] customer: @MicrosoftHelps Hi, my windows 10 updates always failed, So I reset my laptop, but the res


---
## §4 · Escalation Gate

In [16]:
from pipeline.phase4_escalation import EscalationGate, Action

gate = EscalationGate(confidence_threshold=0.35)

test_cases = [
    ('I got a BSOD blue screen every hour please help asap',          'Windows General',          0.85),
    ('My account was hacked someone changed my password',             'Account & Sign-In',        0.70),
    ('I was double charged on my Microsoft 365 subscription',         'Office & Productivity',    0.65),
    ('Windows update stuck at 35% for 2 hours',                      'Windows Update & Install', 0.72),
    ('How do I turn off automatic updates in Windows 10?',            'Windows General',          0.22),
]

print('Escalation gate smoke test:')
for tweet, intent, conf in test_cases:
    d = gate.decide(tweet, intent, conf)
    icon = '🔴 ESCALATE' if d.action == Action.ESCALATE else '✅ AUTO'
    print(f'  {icon} | {d.reason[:65]}')
    print(f'  └─ {tweet[:70]}')

Escalation gate smoke test:
  🔴 ESCALATE | Escalation trigger detected: BSOD / System Crash
  └─ I got a BSOD blue screen every hour please help asap
  🔴 ESCALATE | Escalation trigger detected: Account Lockout / Hack
  └─ My account was hacked someone changed my password
  🔴 ESCALATE | Escalation trigger detected: Payment / Billing Dispute
  └─ I was double charged on my Microsoft 365 subscription
  ✅ AUTO | No escalation triggers detected; retrieval confidence sufficient
  └─ Windows update stuck at 35% for 2 hours
  🔴 ESCALATE | Retrieval confidence (0.22) is below threshold (0.35) — no ground
  └─ How do I turn off automatic updates in Windows 10?


---
## §5 · Reply Drafter (Llama 3.2 3B + LoRA)

In [13]:
# ──────────────────────────────────────────────────────────────────────────
# §5-TRAIN  Fine-tune LoRA drafter  (run ONCE on A100, ~3-4 hrs, ~45 units)
#
# Switch runtime to A100 before running this cell:
#   Runtime → Change runtime type → A100 GPU
# Then re-run §0 cells (install + drive mount), then run this cell.
#
# Skipped automatically if adapter already on Hub.
# ──────────────────────────────────────────────────────────────────────────
from huggingface_hub import list_models

try:
    hf_token = os.environ.get('HF_TOKEN')
    already_trained = any(True for _ in list_models(author=HF_USERNAME, search='llama32-mshelps-drafter', token=hf_token))
except Exception:
    already_trained = False

if already_trained:
    print(f'✅ {DRAFTER_LORA_ID} already on Hub — skipping training')
else:
    print('🏋️ Building drafter training data...')
    from pipeline.phase1_intake import prepare_thread_for_embedding
    from pipeline.phase5_drafter import ReplyDrafter, get_lora_config, build_drafter_training_dataset
    from data.prepare_golden_set import CLUSTER_TO_CLASS
    from peft import get_peft_model, prepare_model_for_kbit_training
    from trl import SFTTrainer, SFTConfig

    # Build (first_inbound_text, brand_reply) training pairs from thread data
    cluster_col = 'cluster_id' if 'cluster_id' in threads_df.columns else 'cluster_final'
    train_rows = []
    for tid, grp in threads_df.groupby('thread_id'):
        grp = grp.sort_values('turn_pos') if 'turn_pos' in grp.columns else grp
        cust  = grp[grp['inbound'] == True]['text'].tolist()
        brand = grp[grp['inbound'] == False]['text'].tolist()
        if not cust or not brand: continue
        cluster = str(grp[cluster_col].iloc[0]) if cluster_col in grp.columns else 'C1_general_os'
        intent  = CLUSTER_TO_CLASS.get(cluster, 'Windows General')
        context = prepare_thread_for_embedding(grp.to_dict('records'), max_turns=6,
                                               author_key='inbound', text_key='text')
        train_rows.append({'first_inbound_text': cust[0], 'brand_reply': brand[0],
                           'intent_label': intent, 'thread_context': context})

    import pandas as pd
    drafter_train_df = pd.DataFrame(train_rows)
    print(f'Training examples: {len(drafter_train_df)}')

    train_dataset = build_drafter_training_dataset(
        drafter_train_df,
        text_col='first_inbound_text',
        reply_col='brand_reply',
        intent_col='intent_label',
        context_col='thread_context',
        max_samples=3000,
    )
    print('🏋️ Loading base model for LoRA training...')
    drafter_for_train = ReplyDrafter.for_training(DRAFTER_BASE_ID)
    drafter_for_train.model = prepare_model_for_kbit_training(drafter_for_train.model)
    peft_model = get_peft_model(drafter_for_train.model, get_lora_config())
    peft_model.print_trainable_parameters()

    trainer = SFTTrainer(
        model=peft_model,
        args=SFTConfig(
            output_dir='/content/llama32_lora_ckpts',
            num_train_epochs=2,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            learning_rate=2e-4,
            fp16=False,
            bf16=True,
            logging_steps=20,
            save_steps=200,
            max_length=512,
            dataset_text_field='text',
            report_to='none',
        ),
        train_dataset=train_dataset,
    )
    trainer.train()
    ReplyDrafter.push_adapter_to_hub(peft_model, DRAFTER_LORA_ID)
    print(f'✅ LoRA adapter pushed to {DRAFTER_LORA_ID}')

✅ SpongeBob387/llama32-mshelps-drafter already on Hub — skipping training


In [11]:
# Load reply drafter (quantized Llama 3.2 3B + LoRA adapter)
from pipeline.phase5_drafter import ReplyDrafter

drafter = ReplyDrafter.from_pretrained(
    base_model=DRAFTER_BASE_ID,
    lora_adapter=DRAFTER_LORA_ID,
    load_in_4bit=True,
)
print('✅ Drafter loaded')

[ReplyDrafter] Loading base model: meta-llama/Llama-3.2-3B-Instruct


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

[ReplyDrafter] Loading LoRA adapter: SpongeBob387/llama32-mshelps-drafter


adapter_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 18.4MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

[ReplyDrafter] ✅ LoRA adapter loaded (4-bit inference mode)
[ReplyDrafter] ✅ Model ready for inference
✅ Drafter loaded


---
## §6 · Safety Guardrails Smoke Test

In [15]:
from pipeline.phase6_guardrails import apply_guardrails

test_replies = [
    'Hi! Try running sfc /scannow in Command Prompt as admin. https://support.microsoft.com/fix',
    'Please send us your email and password so we can look into this.',
    'Check this fix: https://some-random-site.ru/microsoft-hack',
    'x' * 300,
]
for reply in test_replies:
    r = apply_guardrails(reply)
    icon = '✅' if r['all_safe'] else '⚠️'
    print(f'{icon} fixes={r["applied_fixes"]}')
    print(f'   {r["final_reply"][:100]}')

✅ fixes=[]
   Hi! Try running sfc /scannow in Command Prompt as admin. https://support.microsoft.com/fix
⚠️ fixes=["PII solicitation blocked; replaced with safe fallback. Violations: ['send us your email']"]
   We'd love to help! Please visit https://support.microsoft.com or contact us through our official sup
⚠️ fixes=["Unapproved domains found: ['some-random-site.ru']. Reply flagged for review."]
   Check this fix: https://some-random-site.ru/microsoft-hack
✅ fixes=['Reply truncated from 300 to ≤280 characters.']
   xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx


In [40]:
# Save the working model to Drive so future loads use Drive path, not Hub
import os
MODEL_SAVE_DIR = DRIVE_ROOT + 'models/bertweet_intent/'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
classifier.model.save_pretrained(MODEL_SAVE_DIR)
classifier.tokenizer.save_pretrained(MODEL_SAVE_DIR)
print(f"✅ Saved to {MODEL_SAVE_DIR}")

# Update INTENT_MODEL_ID to Drive path so agent.load() never hits Hub again
INTENT_MODEL_ID = MODEL_SAVE_DIR
print(f"INTENT_MODEL_ID = {INTENT_MODEL_ID}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Saved to /content/drive/MyDrive/Hiver/models/bertweet_intent/
INTENT_MODEL_ID = /content/drive/MyDrive/Hiver/models/bertweet_intent/


---
## §7 · Golden Evaluation Set

In [16]:
import os, pandas as pd

GOLDEN_CSV = GOLDEN_SET_DIR + 'golden_set_250.csv'

if os.path.exists(GOLDEN_CSV):
    golden_df = pd.read_csv(GOLDEN_CSV)
    print(f'✅ Loaded golden set: {len(golden_df)} threads')
else:
    print('Building golden set (first run)...')
    os.makedirs(GOLDEN_SET_DIR, exist_ok=True)
    from data.prepare_golden_set import run_in_colab as build_golden
    golden_df = build_golden(
        threads_csv=DATA_DIR + 'ms_threads.csv',
        first_inbound_csv=DATA_DIR + 'microsofthelps_first_inbound_k9.csv',
        output_dir=GOLDEN_SET_DIR,
    )

print('\nClass distribution:')
print(golden_df['intent_class'].value_counts().to_string())
print(f'\nEscalation threads: {golden_df["is_escalation"].sum()} / {len(golden_df)}')

✅ Loaded golden set: 247 threads

Class distribution:
intent_class
Windows General               68
Windows Update & Install      43
BSOD & Crash / Distress       38
Office & Productivity         30
Account & Sign-In             30
Surface Hardware              20
Xbox & Gaming                 16
Support Channel Navigation     2

Escalation threads: 57 / 247


In [57]:
from pipeline.agent import SupportAgent
import os
#INTENT_MODEL_PATH = "/content/drive/MyDrive/Hiver/models/bertweet_intent_test"
INTENT_MODEL_PATH = f"{HF_USERNAME}/bertweet-mshelps-intent"

tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", normalization=True, use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained(INTENT_MODEL_PATH)
saved_classifier = IntentClassifier(model, tokenizer)

agent = SupportAgent.load(
    intent_model=saved_classifier,
    index_dir=INDEX_DIR,
    drafter_base=DRAFTER_BASE_ID,
    drafter_lora=DRAFTER_LORA_ID,
    log_dir=LOG_DIR,
    verbose=False,
)
#agent.classifier = classifier
print("✅ Agent reloaded with fresh model")

model.safetensors: reconstructing file:   0%|          |  0.00B /  540MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

🚀 Loading Hiver AI Support Agent

[1/3] Loading Intent Classifier...
  Using provided IntentClassifier instance.

[2/3] Loading Hybrid Retrieval Index...
[HybridRetriever] Loading BM25 index from /content/drive/MyDrive/Hiver/index/bm25_index.pkl
[HybridRetriever] Loading FAISS index from /content/drive/MyDrive/Hiver/index/faiss_index.bin
[HybridRetriever] Loading metadata from /content/drive/MyDrive/Hiver/index/thread_metadata.json
[HybridRetriever] Loading embedder: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[HybridRetriever] ✅ Index loaded: 4458 threads

[3/3] Loading Reply Drafter...
[ReplyDrafter] Loading base model: meta-llama/Llama-3.2-3B-Instruct


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[ReplyDrafter] Loading LoRA adapter: SpongeBob387/llama32-mshelps-drafter
[ReplyDrafter] ✅ LoRA adapter loaded (4-bit inference mode)
[ReplyDrafter] ✅ Model ready for inference

✅ SupportAgent ready.

✅ Agent reloaded with fresh model


In [58]:
# ── DEBUG CELL — paste and run this before §8 ────────────────────────────────
import traceback

tweet = "my xbox wont connect to xbox live since yesterday"
print(f"Tweet: {tweet}\n")

# Step 1: Is the classifier inside the agent the working one?
pred = agent.classifier.predict(tweet)
print(f"[Phase 2 direct] label={pred.label}, conf={pred.confidence:.3f}")

# Step 2: Run a single tweet through the agent with full traceback
try:
    out = agent.run(tweet)
    print(f"\n[Agent output]")
    print(f"  intent:     {out.intent}")
    print(f"  confidence: {out.intent_confidence}")
    print(f"  action:     {out.action}")
    print(f"  reply:      {out.public_reply[:100]}")
    print(f"  reason:     {out.reason}")
except Exception as e:
    print(f"\n[EXCEPTION in agent.run()]")
    traceback.print_exc()

# Step 3: Check what's loaded inside the agent
print(f"\n[Agent internals]")
print(f"  classifier type:  {type(agent.classifier)}")
print(f"  drafter type:     {type(agent.drafter)}")
print(f"  retriever type:   {type(agent.retriever)}")
print(f"  drafter is None:  {agent.drafter is None}")

Tweet: my xbox wont connect to xbox live since yesterday

[Phase 2 direct] label=Xbox & Gaming, conf=0.590

[Agent output]
  intent:     Xbox & Gaming
  confidence: 0.5904831290245056
  action:     AUTO_HANDLE
  reply:      @374354 Hey, we're glad to help you out, Negan. How was your experience after reaching out to our su
  reason:     No escalation triggers detected; retrieval confidence sufficient

[Agent internals]
  classifier type:  <class 'pipeline.phase2_intent.IntentClassifier'>
  drafter type:     <class 'pipeline.phase5_drafter.ReplyDrafter'>
  retriever type:   <class 'pipeline.phase3_retrieval.HybridRetriever'>
  drafter is None:  False


---
## §8 · Evaluation Harness

In [17]:
# Load full agent and run on golden set subsample
import os, time
from pipeline.agent import SupportAgent

# os.makedirs(LOG_DIR, exist_ok=True)
# agent = SupportAgent.load(
#     intent_model=INTENT_MODEL_ID,
#     index_dir=INDEX_DIR,
#     drafter_base=DRAFTER_BASE_ID,
#     drafter_lora=DRAFTER_LORA_ID,
#     log_dir=LOG_DIR,
#     verbose=False,
# )

eval_df = golden_df.sample(n=min(SUBSAMPLE_N, len(golden_df)), random_state=SEED).reset_index(drop=True)
print(f'Running inference on {len(eval_df)} examples...')

t0 = time.time()
system_outputs = agent.run_batch(eval_df['text'].tolist(), top_k=TOP_K)
elapsed = time.time() - t0
print(f'✅ Done in {elapsed:.1f}s  ({elapsed/len(eval_df):.2f}s per example)')

Running inference on 247 examples...
  [1/247] Processing tweet…
  [11/247] Processing tweet…
  [21/247] Processing tweet…
  [31/247] Processing tweet…
  [41/247] Processing tweet…
  [51/247] Processing tweet…
  [61/247] Processing tweet…
  [71/247] Processing tweet…
  [81/247] Processing tweet…
  [91/247] Processing tweet…
  [101/247] Processing tweet…
  [111/247] Processing tweet…
  [121/247] Processing tweet…
  [131/247] Processing tweet…
  [141/247] Processing tweet…
  [151/247] Processing tweet…
  [161/247] Processing tweet…
  [171/247] Processing tweet…
  [181/247] Processing tweet…
  [191/247] Processing tweet…
  [201/247] Processing tweet…
  [211/247] Processing tweet…
  [221/247] Processing tweet…
  [231/247] Processing tweet…
  [241/247] Processing tweet…
✅ Done in 986.6s  (3.99s per example)


In [18]:
# Intent classification metrics
from evaluation.metrics import compute_classification_metrics, print_classification_report

y_true_intent = eval_df['intent_class'].tolist()
y_pred_intent = [o.intent for o in system_outputs]

cls_metrics = compute_classification_metrics(y_true_intent, y_pred_intent)
print_classification_report(cls_metrics)
SYSTEM_METRICS = {'classification': cls_metrics}


📊 INTENT CLASSIFICATION METRICS
  Macro F1:      0.9134
  Micro F1:      0.8907
  Weighted F1:   0.8914
  Precision:     0.9337
  Recall:        0.9004
  Samples:       247

Per-class breakdown:
  Account & Sign-In                   F1=0.871  P=0.844  R=0.900
  BSOD & Crash / Distress             F1=0.914  P=1.000  R=0.842
  Office & Productivity               F1=0.967  P=0.967  R=0.967
  Support Channel Navigation          F1=1.000  P=1.000  R=1.000
  Surface Hardware                    F1=0.950  P=0.950  R=0.950
  Windows General                     F1=0.859  P=0.790  R=0.941
  Windows Update & Install            F1=0.850  P=0.919  R=0.791
  Xbox & Gaming                       F1=0.897  P=1.000  R=0.812

Full Classification Report:
                            precision    recall  f1-score   support

         Account & Sign-In       0.84      0.90      0.87        30
   BSOD & Crash / Distress       1.00      0.84      0.91        38
     Office & Productivity       0.97      0.97   

In [19]:
# Escalation metrics
from evaluation.metrics import compute_escalation_metrics, print_escalation_report

y_true_esc = eval_df['is_escalation'].astype(int).tolist()
y_pred_esc = [1 if o.action == 'ESCALATE' else 0 for o in system_outputs]

esc_metrics = compute_escalation_metrics(y_true_esc, y_pred_esc)
print_escalation_report(esc_metrics)
SYSTEM_METRICS['escalation'] = esc_metrics


🚨 ESCALATION DECISION METRICS
  Precision:             0.6790
  Recall:                0.9649
  F1:                    0.7971
  ⚠️  FNR (CRITICAL):    0.0351  ← missed escalations
  FPR:                   0.1368
  Accuracy:              0.8866
  TP / FP / FN / TN:    55 / 26 / 2 / 164
  True escalations:      57 / 247


In [20]:
# LLM-as-judge (Groq Llama 3.3 70B) — evaluated on 50 examples
import os
from evaluation.llm_judge import LLMJudge

judge = LLMJudge(groq_api_key=os.environ['GROQ_API_KEY'])

judge_sample = eval_df.sample(n=min(50, len(eval_df)), random_state=SEED)
judge_idx    = [eval_df.index.get_loc(i) for i in judge_sample.index]
judge_outs   = [system_outputs[i] for i in judge_idx]

judge_records = [
    {'tweet': row['text'], 'reply': out.public_reply,
     'intent': out.intent, 'escalation_decision': out.action,
     'escalation_reason': out.reason}
    for (_, row), out in zip(judge_sample.iterrows(), judge_outs)
]

judge_scores = judge.judge_batch(judge_records)
judge_agg    = judge.aggregate_scores(judge_scores)
print('\nLLM Judge aggregate scores:')
for k, v in judge_agg.items(): print(f'  {k}: {v}')
SYSTEM_METRICS['judge'] = judge_agg

[LLMJudge] Evaluating 50 examples with openai/gpt-oss-120b…
  [10/50] avg_overall=2.45  errors=0
  [20/50] avg_overall=2.64  errors=0
  [30/50] avg_overall=2.64  errors=0
  [40/50] avg_overall=2.61  errors=0
[LLMJudge] API error (attempt 1): Error code: 400 - {'error': {'message': "Failed to generate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': 'max completion tokens reached before generating a valid document'}}. Retrying in 2.0s…
[LLMJudge] API error (attempt 2): Error code: 400 - {'error': {'message': "Failed to generate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': 'max completion tokens reached before generating a valid document'}}. Retrying in 2.0s…
[LLMJudge] API error (attempt 1): Error code: 400 - {'error': {'message': "Failed to generate JSON. Pl

In [23]:
# Human-LLM agreement analysis (50 pre-baked human annotations)
from evaluation.human_agreement import compute_full_agreement, print_agreement_report, HUMAN_ANNOTATIONS

n = min(len(HUMAN_ANNOTATIONS), len(judge_scores))
llm_by_dim = {
    'grounding':     [s.grounding     for s in judge_scores[:n]],
    'tone':          [s.tone          for s in judge_scores[:n]],
    'actionability': [s.actionability for s in judge_scores[:n]],
    'safety':        [s.safety        for s in judge_scores[:n]],
}
agreement = compute_full_agreement(llm_by_dim, HUMAN_ANNOTATIONS[:n])
print_agreement_report(agreement)
SYSTEM_METRICS['agreement'] = agreement


🤝 HUMAN–LLM AGREEMENT ANALYSIS
  Examples evaluated: 50

  Dimension: GROUNDING
    Quadratic Kappa: 0.0661  (>0.6 = substantial)
    Linear Kappa:    0.0090
    Pearson r:       0.1628  (p=0.2586)
    Exact Match:     8.0%
    Off-by-One:      40.0%
    Human mean:      3.620  |  LLM mean: 1.940

  Dimension: TONE
    Quadratic Kappa: -0.0270  (>0.6 = substantial)
    Linear Kappa:    0.0124
    Pearson r:       -0.0518  (p=0.7209)
    Exact Match:     26.0%
    Off-by-One:      54.0%
    Human mean:      4.300  |  LLM mean: 3.140

  Dimension: ACTIONABILITY
    Quadratic Kappa: 0.1051  (>0.6 = substantial)
    Linear Kappa:    0.0936
    Pearson r:       0.1790  (p=0.2135)
    Exact Match:     24.0%
    Off-by-One:      50.0%
    Human mean:      3.420  |  LLM mean: 2.040

  Dimension: SAFETY
    Quadratic Kappa: -0.0478  (>0.6 = substantial)
    Linear Kappa:    -0.0551
    Pearson r:       -0.1281  (p=0.3752)
    Exact Match:     26.0%
    Off-by-One:      56.0%
    Human mean:   

---
## §9 · Baseline Comparisons

In [24]:
# Baseline 0: majority class + canned reply + keyword escalation
from evaluation.baselines import MajorityClassBaseline
from evaluation.metrics import compute_classification_metrics, compute_escalation_metrics

b0 = MajorityClassBaseline()
b0_out = b0.run_batch(eval_df['text'].tolist())

B0_METRICS = {
    'classification': compute_classification_metrics(y_true_intent, [o.intent for o in b0_out]),
    'escalation':     compute_escalation_metrics(y_true_esc, [1 if o.action == 'ESCALATE' else 0 for o in b0_out]),
}
print(f'Baseline 0 — Macro F1: {B0_METRICS["classification"]["macro_f1"]:.4f}  '
      f'Escalation Recall: {B0_METRICS["escalation"]["recall"]:.4f}')

Baseline 0 — Macro F1: 0.0540  Escalation Recall: 0.8070


In [25]:
# Baseline 1 classification: TF-IDF + Logistic Regression
from evaluation.baselines import TFIDFLogisticBaseline
from data.prepare_golden_set import CLUSTER_TO_CLASS

fi = first_inbound.copy()
cluster_col = 'cluster_id' if 'cluster_id' in fi.columns else 'cluster_final'
fi['intent_class'] = fi[cluster_col].map(CLUSTER_TO_CLASS).fillna('Windows General')

# Exclude golden set threads from training
golden_ids = set(golden_df['thread_id'].astype(str))
fi_train   = fi[~fi['thread_id'].astype(str).isin(golden_ids)]

b1_clf = TFIDFLogisticBaseline()
b1_clf.fit(fi_train['text'].tolist(), fi_train['intent_class'].tolist())
b1_intent_preds = b1_clf.predict(eval_df['text'].tolist())

b1_cls = compute_classification_metrics(y_true_intent, b1_intent_preds)
print(f'Baseline 1 (TF-IDF+LR) — Macro F1: {b1_cls["macro_f1"]:.4f}')

[TFIDFLogisticBaseline] ✅ Fitted on 4211 examples.
Baseline 1 (TF-IDF+LR) — Macro F1: 0.9068


In [26]:
# Baseline 1 reply + escalation: zero-shot Groq Llama (no RAG)
import os
from evaluation.baselines import ZeroShotGroqBaseline

b1_groq = ZeroShotGroqBaseline(groq_api_key=os.environ['GROQ_API_KEY'])

# Run on the same 50-example judge subset
b1_outs = b1_groq.run_batch(
    judge_sample['text'].tolist(),
    intents=b1_intent_preds[:len(judge_sample)],
)

b1_esc_true  = [y_true_esc[eval_df.index.get_loc(i)] for i in judge_sample.index]
b1_esc_preds = [1 if o.action == 'ESCALATE' else 0 for o in b1_outs]
b1_esc       = compute_escalation_metrics(b1_esc_true, b1_esc_preds)

b1_judge_records = [
    {'tweet': row['text'], 'reply': out.public_reply,
     'intent': out.intent, 'escalation_decision': out.action}
    for (_, row), out in zip(judge_sample.iterrows(), b1_outs)
]
b1_judge_scores = judge.judge_batch(b1_judge_records)
b1_judge_agg    = judge.aggregate_scores(b1_judge_scores)

B1_METRICS = {'classification': b1_cls, 'escalation': b1_esc, 'judge': b1_judge_agg}

[LLMJudge] Evaluating 50 examples with openai/gpt-oss-120b…
  [10/50] avg_overall=1.38  errors=0
  [20/50] avg_overall=1.44  errors=0
  [30/50] avg_overall=1.43  errors=0
  [40/50] avg_overall=1.42  errors=0
  [50/50] avg_overall=1.43  errors=0
[LLMJudge] ✅ Done. avg_overall=1.425  errors=0/50


In [27]:
# Print comparison table
from evaluation.baselines import print_baseline_comparison

print_baseline_comparison(
    system_metrics=   {**SYSTEM_METRICS['classification'], 'escalation': SYSTEM_METRICS['escalation']},
    baseline0_metrics={**B0_METRICS['classification'],    'escalation': B0_METRICS['escalation']},
    baseline1_metrics={**B1_METRICS['classification'],    'escalation': B1_METRICS['escalation']},
    judge_system=     SYSTEM_METRICS.get('judge'),
    judge_baseline0=  None,
    judge_baseline1=  B1_METRICS.get('judge'),
    )



📊 RESULTS vs. BASELINES
Metric                           Baseline 0   Baseline 1   Our System
─────────────────────────────────────────────────────────────────────

  ── Intent Classification ──
  Macro F1                           0.0540       0.9068       0.9134
  Micro F1                           0.2753       0.8947       0.8907
  Weighted F1                        0.1189       0.8951       0.8914

  ── Escalation Decision ──
  Precision                          0.8846       0.0000       0.6790
  Recall                             0.8070       0.0000       0.9649
  FNR (⚠️ lower=better)              0.1930       1.0000       0.0351

  ── LLM-as-Judge (mean scores 1–5) ──
  Grounding                               —       1.0000       1.9400
  Actionability                           —       1.0000       2.0400
  Tone                                    —       1.0000       3.1400
  Overall                                 —       1.4250       2.6650
───────────────────────────────────

---
## §10 · Failure Analysis

In [28]:
import pandas as pd

results_rows = []
for i, (idx, row) in enumerate(eval_df.iterrows()):
    out = system_outputs[i]
    results_rows.append({
        'thread_id':         row.get('thread_id', idx),
        'tweet':             row['text'],
        'true_intent':       row['intent_class'],
        'pred_intent':       out.intent,
        'true_escalation':   int(row['is_escalation']),
        'pred_escalation':   1 if out.action == 'ESCALATE' else 0,
        'reply':             out.public_reply,
        'escalation_reason': out.reason,
        'intent_correct':    row['intent_class'] == out.intent,
        'esc_correct':       int(row['is_escalation']) == (1 if out.action == 'ESCALATE' else 0),
    })
results_df = pd.DataFrame(results_rows)

print('=' * 60)
print('FAILURE ANALYSIS — TOP 5 FAILURE MODES')
print('=' * 60)

fm1 = results_df[~results_df['intent_correct'] & (results_df['pred_intent'] == 'Windows General')]
print(f'\n[FM1] Predicted Windows General incorrectly: {len(fm1)} cases')
if not fm1.empty:
    ex = fm1.iloc[0]
    print(f'  True={ex["true_intent"]} | Tweet: {ex["tweet"][:80]}')

fm2 = results_df[(results_df['true_escalation'] == 1) & (results_df['pred_escalation'] == 0)]
print(f'\n[FM2] Missed escalations (False Negatives): {len(fm2)} cases  ← most critical')
if not fm2.empty:
    ex = fm2.iloc[0]
    print(f'  Reply given: {ex["reply"][:80]}')
    print(f'  Tweet: {ex["tweet"][:80]}')

fm3 = results_df[(results_df['true_escalation'] == 0) & (results_df['pred_escalation'] == 1)]
print(f'\n[FM3] Over-escalation (False Positives): {len(fm3)} cases')
if not fm3.empty:
    ex = fm3.iloc[0]
    print(f'  Reason: {ex["escalation_reason"][:80]}')

fm4 = results_df[results_df['true_intent'] == 'Support Channel Navigation']
fm4_wrong = fm4[fm4['pred_intent'] != 'Support Channel Navigation']
print(f'\n[FM4] C3 routing class misclassified: {len(fm4_wrong)} / {len(fm4)} cases')

fm5 = results_df[
    ~results_df['intent_correct'] &
    ((results_df['true_intent'] == 'BSOD & Crash / Distress') |
     (results_df['pred_intent'] == 'BSOD & Crash / Distress'))
]
print(f'\n[FM5] BSOD/Distress boundary confusion: {len(fm5)} cases')
if not fm5.empty:
    ex = fm5.iloc[0]
    print(f'  True={ex["true_intent"]} | Pred={ex["pred_intent"]}')
    print(f'  Tweet: {ex["tweet"][:80]}')

FAILURE ANALYSIS — TOP 5 FAILURE MODES

[FM1] Predicted Windows General incorrectly: 17 cases
  True=BSOD & Crash / Distress | Tweet: @MicrosoftHelps Hi. I need help with a return/refund for @123127. Tried reaching

[FM2] Missed escalations (False Negatives): 2 cases  ← most critical
  Reply given: Hi! 👋 You can reach us via:
• 🌐 https://support.microsoft.com
• 💬 Virtual Agent:
  Tweet: Try to update @116230 to Creator:
"You can contact Microsoft support for help wi

[FM3] Over-escalation (False Positives): 26 cases
  Reason: Intent class 'BSOD & Crash / Distress' is a high-risk escalation category

[FM4] C3 routing class misclassified: 0 / 2 cases

[FM5] BSOD/Distress boundary confusion: 6 cases
  True=BSOD & Crash / Distress | Pred=Windows General
  Tweet: @MicrosoftHelps Hi. I need help with a return/refund for @123127. Tried reaching


---
## §11 · Live Demo (< 15 min reproducibility checkpoint)

In [33]:
import time

demo_tweets = [
    'My Windows 10 update has been stuck at 30% for 3 hours, tried restarting twice',
    'Outlook keeps crashing every time I try to open an attachment',
    'Surface Pro pen stopped working after last Windows update, checked battery already',
    'I got a blue screen of death and now my PC wont boot at all PLEASE HELP ASAP',
    'Someone hacked my Microsoft account and changed the recovery email, Im locked out',
]

print('=' * 70)
print('HIVER AI SUPPORT AGENT — LIVE DEMO')
print('=' * 70)

t_demo = time.time()
for i, tweet in enumerate(demo_tweets, 1):
    out = agent.run(tweet)
    icon = '🔴 ESCALATE' if out.action == 'ESCALATE' else '✅ AUTO_HANDLE'
    print(f'\n── {i} ───────────────────────────────────────────────────────────')
    print(f'📨 {tweet}')
    print(f'🏷️  Intent:  {out.intent} (conf={out.intent_confidence:.2f})')
    print(f'⚡ {icon}')
    if out.action == 'ESCALATE':
        print(f'📋 Reason:  {out.reason}')
    print(f'💬 Reply:   {out.public_reply}')

demo_elapsed = time.time() - t_demo
total_elapsed = elapsed + demo_elapsed

print(f'\n{"=" * 70}')
print(f'Demo time:       {demo_elapsed:.1f}s')
print(f'Full eval time:  {elapsed:.1f}s ({len(eval_df)} examples)')
print(f'Total runtime:   {total_elapsed:.1f}s')
print(f'15-min check:    {"✅ PASS" if total_elapsed < 900 else "❌ EXCEEDS 15 min"}')

HIVER AI SUPPORT AGENT — LIVE DEMO

── 1 ───────────────────────────────────────────────────────────
📨 My Windows 10 update has been stuck at 30% for 3 hours, tried restarting twice
🏷️  Intent:  Windows Update & Install (conf=0.92)
⚡ ✅ AUTO_HANDLE
💬 Reply:   @179912 Hi there! Are you still trying to download and install the latest updates on your device? Let us know if you're still having issues.

── 2 ───────────────────────────────────────────────────────────
📨 Outlook keeps crashing every time I try to open an attachment
🏷️  Intent:  Office & Productivity (conf=0.66)
⚡ 🔴 ESCALATE
📋 Reason:  Escalation trigger detected: BSOD / System Crash
💬 Reply:   @158370 Hi there! Were we able to help? Please send us a DM if you need further assistance. We're here for you! https://t.co/3qcAsLFkaY

── 3 ───────────────────────────────────────────────────────────
📨 Surface Pro pen stopped working after last Windows update, checked battery already
🏷️  Intent:  Surface Hardware (conf=0.71)
⚡ ✅ AUTO_H

## §12 · Interactive Live Chat
Run this cell to talk to the agent directly!

In [34]:
print("=" * 60)
print("🤖 HIVER AI SUPPORT AGENT — LIVE CHAT")
print("Type 'exit' or 'quit' to stop.")
print("=" * 60)

while True:
    user_input = input("\n👤 You: ")
    if user_input.strip().lower() in ['exit', 'quit']:
        print("\nEnding chat. Goodbye!")
        break

    if not user_input.strip():
        continue

    try:
        out = agent.run(user_input)
        icon = '🔴 ESCALATE' if out.action == 'ESCALATE' else '✅ AUTO_HANDLE'

        print(f"\n🏷️  Intent: {out.intent} (conf={out.intent_confidence:.2f})")
        print(f"⚡ Action: {icon}")
        if out.action == 'ESCALATE':
            print(f"📋 Reason: {out.reason}")
        print(f"💬 Agent: {out.public_reply}")
    except Exception as e:
        print(f"\n⚠️ Error processing request: {e}")

🤖 HIVER AI SUPPORT AGENT — LIVE CHAT
Type 'exit' or 'quit' to stop.

👤 You: hi

🏷️  Intent: Windows General (conf=0.90)
⚡ Action: ✅ AUTO_HANDLE
💬 Agent: @230627 Hi there! To answer your question, it is best that you post your concern on our Community forum: https://t.co/1Pg9pOSEY0.

👤 You: my windows is dead

🏷️  Intent: Windows Update & Install (conf=0.92)
⚡ Action: ✅ AUTO_HANDLE
💬 Agent: @643802 Hello, Matt. We'd like to follow up on the status of your concern. Feel free to reach out if you need further assistance.

👤 You: how to uninstall windows

🏷️  Intent: Windows Update & Install (conf=0.92)
⚡ Action: ✅ AUTO_HANDLE
💬 Agent: @266670 Hello! We're here to help, Curtis. To delete Windows.old, we suggest running the Disk cleanup. Here's how: https://t.co/vA7VaqhQfv. Let us know if you have further questions.

👤 You: surface pro pen stopped working

🏷️  Intent: Surface Hardware (conf=0.72)
⚡ Action: ✅ AUTO_HANDLE
💬 Agent: @795512 Hello! 👋 How’s it going now? 😃 Please let us know about

In [35]:
# ==============================================================================
# 🔍 SAVE/LOAD COMPARISON DIAGNOSTIC
# ==============================================================================
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

test_tweet = "my xbox wont connect to xbox live since yesterday"
save_path = "/content/drive/MyDrive/Hiver/models/bertweet_intent_test"

# 1. Save the working in-memory model and tokenizer
print("1. Saving in-memory classifier to Drive...")
classifier.model.save_pretrained(save_path)
classifier.tokenizer.save_pretrained(save_path)

# 2. Reload directly using raw HuggingFace classes
print("2. Reloading model and tokenizer from Drive...")
reloaded_tok = AutoTokenizer.from_pretrained(save_path, use_fast=False)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(save_path).to(classifier.device)
reloaded_model.eval()

# 3. Compare Tokenizer Outputs
tok_in_memory = classifier.tokenizer(test_tweet, return_tensors="pt")["input_ids"]
tok_reloaded  = reloaded_tok(test_tweet, return_tensors="pt")["input_ids"]

print("\n--- TOKENIZER CHECK ---")
print(f"Tokenizer class (in-memory): {type(classifier.tokenizer).__name__}")
print(f"Tokenizer class (reloaded) : {type(reloaded_tok).__name__}")
print(f"Tokens (in-memory): {tok_in_memory.tolist()}")
print(f"Tokens (reloaded) : {tok_reloaded.tolist()}")
print(f"Tokens match?     : {torch.equal(tok_in_memory, tok_reloaded)}")

# 4. Compare Model Weights
print("\n--- WEIGHTS CHECK ---")
w_in_memory = classifier.model.classifier.out_proj.weight.data
w_reloaded  = reloaded_model.classifier.out_proj.weight.data
weight_diff = torch.abs(w_in_memory - w_reloaded).max().item()
print(f"Max weight difference in classification head: {weight_diff}")

# 5. Compare Model Outputs
with torch.no_grad():
    logits_in_memory = classifier.model(tok_in_memory.to(classifier.device)).logits
    logits_reloaded  = reloaded_model(tok_reloaded.to(classifier.device)).logits
    # Cross-test: run reloaded tokens through in-memory model, and vice-versa
    logits_cross     = reloaded_model(tok_in_memory.to(classifier.device)).logits

print("\n--- PREDICTION CHECK ---")
print(f"In-memory pred : {classifier.model.config.id2label[logits_in_memory.argmax().item()]} (argmax={logits_in_memory.argmax().item()})")
print(f"Reloaded pred  : {reloaded_model.config.id2label[logits_reloaded.argmax().item()]} (argmax={logits_reloaded.argmax().item()})")
print(f"Cross-test pred: {reloaded_model.config.id2label[logits_cross.argmax().item()]} (using in-memory tokens)")
print(f"Config id2label: {reloaded_model.config.id2label}")

1. Saving in-memory classifier to Drive...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2. Reloading model and tokenizer from Drive...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- TOKENIZER CHECK ---
Tokenizer class (in-memory): BertweetTokenizer
Tokenizer class (reloaded) : BertweetTokenizer
Tokens (in-memory): [[0, 23, 9820, 1860, 4925, 9, 9820, 294, 324, 873, 2]]
Tokens (reloaded) : [[0, 592, 469, 577, 607, 641, 314, 724, 641, 608, 552, 715, 641, 608, 608, 515, 715, 552, 543, 322, 577, 607, 641, 314, 836, 510, 717, 853, 423, 510, 608, 715, 853, 460, 515, 423, 543, 515, 855, 541, 527, 469, 2]]
Tokens match?     : False

--- WEIGHTS CHECK ---
Max weight difference in classification head: 0.0

--- PREDICTION CHECK ---
In-memory pred : Xbox & Gaming (argmax=6)
Reloaded pred  : Windows General (argmax=0)
Cross-test pred: Xbox & Gaming (using in-memory tokens)
Config id2label: {0: 'Windows General', 1: 'Windows Update & Install', 2: 'Office & Productivity', 3: 'Account & Sign-In', 4: 'Surface Hardware', 5: 'BSOD & Crash / Distress', 6: 'Xbox & Gaming', 7: 'Support Channel Navigation'}


In [36]:
# ── Verification: Load from saved weights with base tokenizer ────────────────
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from pipeline.phase2_intent import IntentClassifier

save_path = "/content/drive/MyDrive/Hiver/models/bertweet_intent_test"  # or INTENT_MODEL_ID from Hub

# 1. Instantiate using the fixed logic
base_tok = AutoTokenizer.from_pretrained("vinai/bertweet-base", normalization=True, use_fast=False)
saved_model = AutoModelForSequenceClassification.from_pretrained(save_path)
fixed_classifier = IntentClassifier(saved_model, base_tok)

# 2. Test the loaded model!
test_tweets = [
    "my xbox wont connect to xbox live since yesterday",
    "excel keeps freezing whenever i open a large spreadsheet",
    "my surface pro pen stopped working",
    "how do i contact microsoft support by phone?"
]

print("Fixed Loaded Classifier Predictions:")
for t in test_tweets:
    pred = fixed_classifier.predict(t)
    print(f"  {pred.label:<30} (conf={pred.confidence:.2f})  |  {t[:45]}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Fixed Loaded Classifier Predictions:
  Xbox & Gaming                  (conf=0.60)  |  my xbox wont connect to xbox live since yeste
  Office & Productivity          (conf=0.65)  |  excel keeps freezing whenever i open a large 
  Surface Hardware               (conf=0.73)  |  my surface pro pen stopped working
  Support Channel Navigation     (conf=0.66)  |  how do i contact microsoft support by phone?


In [48]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from pipeline.phase2_intent import IntentClassifier
from pipeline.agent import SupportAgent

# 1. Load the fine-tuned classifier from Drive/Hub with the base tokenizer
INTENT_MODEL_PATH = "/content/drive/MyDrive/Hiver/models/bertweet_intent_test"

tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", normalization=True, use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained(INTENT_MODEL_PATH)
saved_classifier = IntentClassifier(model, tokenizer)

# 2. Pass directly to SupportAgent.load
agent = SupportAgent.load(
    intent_model=saved_classifier,   # <-- passes the loaded saved model directly
    index_dir=INDEX_DIR,
    drafter_base=DRAFTER_BASE_ID,
    drafter_lora=DRAFTER_LORA_ID,
    log_dir=LOG_DIR,
    verbose=False,
)

# 3. Test!
test_out = agent.run("my xbox wont connect to xbox live since yesterday")
print(f"✅ Verified Intent: {test_out.intent} (conf={test_out.intent_confidence:.2f})")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

🚀 Loading Hiver AI Support Agent

[1/3] Loading Intent Classifier...
  Using provided IntentClassifier instance.

[2/3] Loading Hybrid Retrieval Index...
[HybridRetriever] Loading BM25 index from /content/drive/MyDrive/Hiver/index/bm25_index.pkl
[HybridRetriever] Loading FAISS index from /content/drive/MyDrive/Hiver/index/faiss_index.bin
[HybridRetriever] Loading metadata from /content/drive/MyDrive/Hiver/index/thread_metadata.json
[HybridRetriever] Loading embedder: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[HybridRetriever] ✅ Index loaded: 4458 threads

[3/3] Loading Reply Drafter...
[ReplyDrafter] Loading base model: meta-llama/Llama-3.2-3B-Instruct


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[ReplyDrafter] Loading LoRA adapter: SpongeBob387/llama32-mshelps-drafter
[ReplyDrafter] ✅ LoRA adapter loaded (4-bit inference mode)
[ReplyDrafter] ✅ Model ready for inference

✅ SupportAgent ready.

✅ Verified Intent: Xbox & Gaming (conf=0.60)
